In [1]:
!pip install pandas numpy scikit-learn matplotlib pyarrow -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import os,json,csv,time,random,logging

In [11]:
random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

In [17]:
# Generating dataset
from datetime import date, datetime,timedelta
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })
df = pd.DataFrame(records)
df['revenue'] = df['quantity']*df['unit_price'].round(2)
df.head()


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,USB Hub,Laptop,South,50,844.13,2024-01-25,completed,42206.50
1,ORD-00001,Speaker,Monitor,East,7,828.18,2024-05-09,completed,5797.26
2,ORD-00002,Webcam,Keyboard,South,27,461.70,2024-02-07,completed,12465.90
3,ORD-00003,Webcam,Mouse,North,44,402.74,2024-06-25,completed,17720.56
4,ORD-00004,Laptop,Headset,East,4,237.01,2024-04-17,pending,948.04


In [18]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: SQL: SELECT, WHERE, GROUP BY')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: SQL: SELECT, WHERE, GROUP BY

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.474800    507.536710  12907.569752
std       14.473227    284.635991  11074.194730
min        1.000000     10.010000     11.630000
25%       13.000000    263.212500   3684.605000
50%       26.000000    508.855000   9701.380000
75%       38.000000    754.462500  19634.530000
max       50.000000    999.970000  49937.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  categor

In [32]:
# Cell 4: Core Implementation — SQL: SELECT, WHERE, GROUP BY

import sqlite3

# Load the DataFrame into an in-memory SQLite database
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

# --- SELECT: retrieve specific columns ---
print('=== 1. SELECT — Retrieve specific columns ===')
result = pd.read_sql('SELECT order_id, product, revenue FROM orders LIMIT 5', conn)
print(result)

# --- WHERE: filter rows by condition ---
print('\n=== 2. WHERE — Filter completed orders with revenue > 5000 ===')
result = pd.read_sql('''
    SELECT order_id, product, region, revenue, status
    FROM orders
    WHERE status = 'completed' AND revenue > 5000
    ORDER BY revenue DESC
    LIMIT 10
''', conn)
print(result)

# --- COUNT with WHERE ---
print('\n=== 3. COUNT — Orders per status ===')
result = pd.read_sql('''
    SELECT status, COUNT(*) AS order_count
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC
''', conn)
print(result)

# --- GROUP BY: aggregate by region ---
print('\n=== 4. GROUP BY — Revenue summary by region ===')
result = pd.read_sql('''
    SELECT region,
           COUNT(*) AS order_count,
           ROUND(SUM(revenue), 2) AS total_revenue,
           ROUND(AVG(revenue), 2) AS avg_revenue,
           MIN(revenue) AS min_revenue,
           MAX(revenue) AS max_revenue
    FROM orders
    GROUP BY region
    ORDER BY total_revenue DESC
''', conn)
print(result)

# --- GROUP BY with HAVING: filter groups ---
print('\n=== 5. HAVING — Products with avg revenue > 10000 ===')
result = pd.read_sql('''
    SELECT product,
           COUNT(*) AS order_count,
           ROUND(AVG(revenue), 2) AS avg_revenue,
           ROUND(SUM(revenue), 2) AS total_revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY product
    HAVING AVG(revenue) > 10000
    ORDER BY avg_revenue DESC
''', conn)
print(result)

# --- Multi-column GROUP BY ---
print('\n=== 6. Multi-column GROUP BY — Region + Product ===')
result = pd.read_sql('''
    SELECT region, product,
           COUNT(*) AS orders,
           ROUND(SUM(revenue), 2) AS total_rev
    FROM orders
    WHERE status = 'completed'
    GROUP BY region, product
    ORDER BY total_rev DESC
    LIMIT 10
''', conn)
print(result)

# --- Date filtering with WHERE ---
print('\n=== 7. Date Filtering — Q1 2024 orders ===')
result = pd.read_sql('''
    SELECT COUNT(*) AS q1_orders,
           ROUND(SUM(revenue), 2) AS q1_revenue
    FROM orders
    WHERE order_date BETWEEN '2024-01-01' AND '2024-03-31'
      AND status = 'completed'
''', conn)
print(result)

conn.close()
print('\n-- SQL: SELECT, WHERE, GROUP BY implementation complete')

=== 1. SELECT — Retrieve specific columns ===
    order_id  product   revenue
0  ORD-00000  USB Hub  42206.50
1  ORD-00001  Speaker   5797.26
2  ORD-00002   Webcam  12465.90
3  ORD-00003   Webcam  17720.56
4  ORD-00004   Laptop    948.04

=== 2. WHERE — Filter completed orders with revenue > 5000 ===
    order_id    product region   revenue     status
0  ORD-04187    Headset  North  49558.50  completed
1  ORD-00084    Headset   East  48523.50  completed
2  ORD-05146     Tablet   East  48523.23  completed
3  ORD-05599  Desk Lamp   East  48040.09  completed
4  ORD-04606   Keyboard  South  47912.16  completed
5  ORD-09766     Webcam   West  47605.50  completed
6  ORD-09635  Desk Lamp  South  47542.74  completed
7  ORD-05721    USB Hub  South  47375.50  completed
8  ORD-06132    Monitor  South  47195.00  completed
9  ORD-08205  Desk Lamp  North  46948.86  completed

=== 3. COUNT — Orders per status ===
      status  order_count
0  completed         7027
1    pending         1481
2  cancell

In [33]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: SQL: SELECT, WHERE, GROUP BY')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: SQL: SELECT, WHERE, GROUP BY
Input rows: 10000
Processing complete

Revenue by Region:
region
South    32590036.50
East     32386613.37
West     32134680.93
North    31964366.72
Name: revenue, dtype: float64


In [49]:
# Cell 6: Self-Check — SQL SELECT WHERE GROUP BY (FIXED)

import sqlite3

print('=' * 50)
print('SELF-CHECK — SQL SELECT WHERE GROUP BY')
print('=' * 50)

# Recreate connection for validation
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

checks = [
    ('Connection exists',
     lambda: conn is not None),

    ('Orders table has >= 100 rows',
     lambda: pd.read_sql("SELECT COUNT(*) as c FROM orders", conn)['c'][0] >= 100),

    ('GROUP BY region returns multiple rows',
     lambda: len(pd.read_sql("SELECT region FROM orders GROUP BY region", conn)) > 1),

    ('WHERE revenue > 100 returns results',
     lambda: pd.read_sql("SELECT * FROM orders WHERE revenue > 100", conn).shape[0] > 0),

    ('SUM aggregation is positive',
     lambda: pd.read_sql("SELECT SUM(revenue) as s FROM orders", conn)['s'][0] > 0),

    ('ORDER BY + LIMIT returns 5 rows',
     lambda: len(pd.read_sql("SELECT * FROM orders ORDER BY revenue DESC LIMIT 5", conn)) == 5),

    ('GROUP BY product works',
     lambda: len(pd.read_sql("SELECT product, COUNT(*) FROM orders GROUP BY product", conn)) > 1),

    ('Table has at least 100 rows (duplicate check)',
     lambda: pd.read_sql("SELECT COUNT(*) as c FROM orders", conn)['c'][0] >= 100),
]

passed = 0

for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result:
            passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')

if passed == len(checks):
    print('\n All checks passed! Ready for submission.')
else:
    print('\n Some checks failed. Review your implementation.')
    
conn.close()

SELF-CHECK — SQL SELECT WHERE GROUP BY
  PASS: Connection exists
  PASS: Orders table has >= 100 rows
  PASS: GROUP BY region returns multiple rows
  PASS: WHERE revenue > 100 returns results
  PASS: SUM aggregation is positive
  PASS: ORDER BY + LIMIT returns 5 rows
  PASS: GROUP BY product works
  PASS: Table has at least 100 rows (duplicate check)

8/8 self-checks passed

 All checks passed! Ready for submission.


In [50]:
import sqlite3

conn = sqlite3.connect(':memory:')

# Products table
products_data = []
for i in range(20):
    products_data.append({
        'product_id': f'P{i:03d}',
        'name': f'Product_{i}',
        'brand': random.choice(['BrandA', 'BrandB', 'BrandC']),
        'unit_cost': round(random.uniform(5, 500), 2)
    })

products_df = pd.DataFrame(products_data)
products_df.to_sql('products', conn, index=False, if_exists='replace')

# Sales fact table
sales_data = []
channels = ['Online', 'Store', 'Marketplace']

for i in range(15000):
    sales_data.append({
        'sale_id': f'S{i:05d}',
        'date': str(date(2024,1,1) + timedelta(days=random.randint(0, 180))),
        'region': random.choice(['North','South','East','West']),
        'store_id': f'ST{random.randint(1,50):03d}',
        'product_id': random.choice(products_df['product_id']),
        'category': random.choice(['Electronics','Accessories','Home']),
        'quantity': random.randint(1, 20),
        'unit_price': round(random.uniform(20, 1000), 2),
        'discount_pct': random.choice([0,5,10,15,20,25,30]),
        'channel': random.choice(channels)
    })

sales_df = pd.DataFrame(sales_data)
sales_df.to_sql('sales_fact', conn, index=False, if_exists='replace')

print("✓ sales_fact and products tables created")

✓ sales_fact and products tables created


In [51]:
query1 = """
SELECT 
    region,
    strftime('%Y-%m', date) AS month,
    ROUND(SUM(quantity * unit_price * (1 - discount_pct/100)), 2) AS revenue
FROM sales_fact
WHERE date BETWEEN '2024-01-01' AND '2024-03-31'
GROUP BY region, month
ORDER BY revenue DESC;
"""

pd.read_sql(query1, conn)

,region,month,revenue
0,South,2024-03,3591782.88
1,North,2024-01,3552524.47
2,East,2024-03,3538481.83
3,South,2024-01,3482719.34
4,East,2024-02,3466334.22
5,West,2024-01,3459676.44
6,West,2024-03,3414157.75
7,South,2024-02,3305299.52
8,East,2024-01,3292047.93
9,North,2024-03,3223639.74


In [52]:
query2 = """
SELECT *
FROM (
    SELECT 
        sf.product_id,
        p.name,
        ROUND(SUM(sf.quantity * sf.unit_price * (1 - sf.discount_pct/100)), 2) AS revenue,
        RANK() OVER (ORDER BY SUM(sf.quantity * sf.unit_price * (1 - sf.discount_pct/100)) DESC) AS rank
    FROM sales_fact sf
    JOIN products p ON sf.product_id = p.product_id
    GROUP BY sf.product_id
)
WHERE rank <= 10;
"""

pd.read_sql(query2, conn)

,product_id,name,revenue,rank
0,P013,Product_13,4245844.57,1
1,P012,Product_12,4242773.91,2
2,P010,Product_10,4219974.40,3
3,P005,Product_5,4170488.17,4
4,P000,Product_0,4150294.35,5
5,P019,Product_19,4148900.59,6
6,P009,Product_9,4134273.55,7
7,P014,Product_14,4093862.93,8
8,P001,Product_1,4084373.62,9
9,P004,Product_4,4045226.44,10


In [53]:
query3 = """
SELECT 
    date,
    ROUND(SUM(quantity * unit_price * (1 - discount_pct/100)), 2) AS daily_revenue,
    COUNT(*) AS orders
FROM sales_fact
GROUP BY date
ORDER BY date;
"""

pd.read_sql(query3, conn).head()

,date,daily_revenue,orders
0,2024-01-01,478093.52,84
1,2024-01-02,464498.65,78
2,2024-01-03,398924.95,75
3,2024-01-04,589121.65,93
4,2024-01-05,355062.20,77


In [54]:
query4 = """
SELECT 
    discount_pct,
    COUNT(*) AS orders,
    ROUND(AVG(quantity * unit_price), 2) AS avg_gross,
    ROUND(AVG(quantity * unit_price * (1 - discount_pct/100)), 2) AS avg_net
FROM sales_fact
GROUP BY discount_pct
ORDER BY discount_pct;
"""

pd.read_sql(query4, conn)

,discount_pct,orders,avg_gross,avg_net
0,0,2192,5285.12,5285.12
1,5,2086,5298.54,5298.54
2,10,2148,5274.99,5274.99
3,15,2169,5442.91,5442.91
4,20,2154,5228.10,5228.10
5,25,2143,5397.98,5397.98
6,30,2108,5383.43,5383.43


In [55]:
query5 = """
SELECT 
    channel,
    ROUND(SUM(quantity * unit_price * (1 - discount_pct/100)), 2) AS revenue,
    RANK() OVER (ORDER BY SUM(quantity * unit_price * (1 - discount_pct/100)) DESC) AS rank
FROM sales_fact
GROUP BY channel;
"""

pd.read_sql(query5, conn)

,channel,revenue,rank
0,Marketplace,27586043.59,1
1,Store,26922866.64,2
2,Online,25442658.91,3


In [56]:
print('=' * 50)
print('TRY-IT-OUT VALIDATION CHECKS')
print('=' * 50)

checks = [

    ('Revenue is positive',
     lambda: pd.read_sql("SELECT SUM(quantity * unit_price) as r FROM sales_fact", conn)['r'][0] > 0),

    ('Discount range valid (0–30)',
     lambda: pd.read_sql("SELECT COUNT(*) as c FROM sales_fact WHERE discount_pct < 0 OR discount_pct > 30", conn)['c'][0] == 0),

    ('At least 10 unique products',
     lambda: pd.read_sql("SELECT COUNT(DISTINCT product_id) as c FROM sales_fact", conn)['c'][0] >= 10),
]

passed = 0

for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result:
            passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} checks passed')

TRY-IT-OUT VALIDATION CHECKS
  PASS: Revenue is positive
  PASS: Discount range valid (0–30)
  PASS: At least 10 unique products

3/3 checks passed
